# Predicción del segmento de proveedores sin órdenes (HU 6.1 y HU 6.2)
**Autor:** Miguel · **Entradas:** `data/processed/purchasing_consolidated.csv`, `data/processed/purchasing_segmentacion_final.csv`, `data/raw/vendor_attributes.csv`

Objetivo: predecir `Segmento` para los 18 proveedores sin órdenes, entrenando con los proveedores que sí tienen segmento.

## Resumen de decisiones
- **Variable objetivo:** `Segmento` (Grandes recurrentes, Pequeños recurrentes, Ocasionales), definida en la segmentación final.
- **Predictoras:** solo variables que existen para los 106 proveedores y que no se calcularon a partir de las órdenes (`CreditRating`, `PreferredVendorStatus`, `ActiveFlag`), más dos atributos probados en la TASK 6.1.2 (`ProvinciaGrupo`, `TieneURL`).
- **Registros de prueba:** los proveedores 1699 y 1700 (`Distribuidora Ejemplo S.A.` y `garfield`, con direcciones en Alemania y Australia, fechas de 2026 y 3 y 1 órdenes por montos ínfimos) se **excluyen del modelado**. La segmentación de origen no se modifica.
- **Validación:** validación cruzada estratificada repetida con la misma semilla para todos los modelos y líneas base.
- **Modelos:** regresión logística y Random Forest, comparados contra dos líneas base.

**Qué hace esta celda:** importa las librerías, fija la semilla (`SEED = 42`, para que los resultados se puedan repetir) y carga dos archivos: el consolidado de los 106 proveedores y la segmentación final de los 88 con órdenes. Los une por `VendorID` y crea la columna `conjunto`.

**Qué significa:** los proveedores con segmento conocido quedan como `entrenamiento` (88) y los que no tienen segmento quedan como `predecir` (18). Las cifras impresas deben coincidir: 106 filas en total, 88 + 18, y la distribución del segmento (con 18 vacíos).

In [16]:
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import RepeatedStratifiedKFold, cross_val_predict
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score
import warnings; warnings.filterwarnings('ignore')

ROOT = Path('..') if Path('../data').exists() else Path('.')
PROC = ROOT / 'data/processed'
SEED = 42
cons = pd.read_csv(PROC/'purchasing_consolidated.csv')
seg  = pd.read_csv(PROC/'purchasing_segmentacion_final.csv')[['VendorID','Segmento']]
df = cons.merge(seg, on='VendorID', how='left')
df['conjunto'] = np.where(df['Segmento'].notna(), 'entrenamiento', 'predecir')
print(df.shape); print(df['conjunto'].value_counts()); print(df['Segmento'].value_counts(dropna=False))

(106, 17)
conjunto
entrenamiento    88
predecir         18
Name: count, dtype: int64
Segmento
Grandes recurrentes     40
Pequeños recurrentes    39
NaN                     18
Ocasionales              9
Name: count, dtype: int64


## TASK 6.1.1 – Auditoría de variables
Regla: una predictora solo sirve si (a) existe para los 106 proveedores y (b) no se derivó de las órdenes, porque el segmento se construyó con `ValorTotalOrdenes` y `NumeroOrdenes`.

**Qué hace la celda siguiente:** arma una tabla con una fila por cada columna del consolidado. Indica si la variable existe para los 106 proveedores (`disponible_106`), cuántos datos tiene, cuántos valores distintos, su tipo y la decisión sobre ella (usar o excluir y el motivo).

**Qué significa:** las columnas de órdenes se excluyen porque con ellas se construyó el segmento (usarlas sería *fuga de datos*) y además están vacías para los 18. Las variables de `ProductVendor` (`ProductosOfrecidos`, `TiempoEntregaPromedio`, `PrecioEstandarPromedio`) tampoco existen para los 18. Solo `CreditRating`, `PreferredVendorStatus` y `ActiveFlag` cumplen ambas condiciones.

In [17]:
aud = pd.DataFrame({'disponible_106': df.notna().all(), 'n_no_nulos': df.notna().sum(), 'n_unicos': df.nunique(), 'tipo': df.dtypes.astype(str)})
motivo = {
 'VendorID':'Identificador (no predice)', 'NombreProveedor':'Identificador (riesgo de ruido/fuga)',
 'CreditRating':'USAR: atributo del proveedor, existe para los 106', 'PreferredVendorStatus':'USAR: existe para los 106', 'ActiveFlag':'USAR (casi constante)',
 'NumeroOrdenes':'EXCLUIR: define el segmento (fuga) y no existe para los 18', 'ValorTotalOrdenes':'EXCLUIR: define el segmento (fuga) y no existe para los 18',
 'GastoPromedioOrden':'EXCLUIR: derivada de órdenes', 'UltimaOrden':'EXCLUIR: derivada de órdenes', 'UnidadesPedidas':'EXCLUIR: derivada de órdenes',
 'ProductosComprados':'EXCLUIR: derivada de órdenes', 'ValorLineas':'EXCLUIR: derivada de órdenes',
 'ProductosOfrecidos':'EXCLUIR: no existe para los 18 (ninguno está en ProductVendor)', 'TiempoEntregaPromedio':'EXCLUIR: no existe para los 18',
 'PrecioEstandarPromedio':'EXCLUIR: no existe para los 18', 'Segmento':'VARIABLE OBJETIVO', 'conjunto':'Auxiliar'}
aud['decision'] = aud.index.map(motivo)
aud

,disponible_106,n_no_nulos,n_unicos,tipo,decision
VendorID,True,106,106,int64,Identificador (no predice)
NombreProveedor,True,106,106,str,Identificador (riesgo de ruido/fuga)
CreditRating,True,106,5,int64,"USAR: atributo del proveedor, existe para los 106"
PreferredVendorStatus,True,106,2,bool,USAR: existe para los 106
ActiveFlag,True,106,2,bool,USAR (casi constante)
NumeroOrdenes,False,88,7,float64,EXCLUIR: define el segmento (fuga) y no existe...
ValorTotalOrdenes,False,88,86,float64,EXCLUIR: define el segmento (fuga) y no existe...
GastoPromedioOrden,False,88,86,float64,EXCLUIR: derivada de órdenes
UltimaOrden,False,88,13,str,EXCLUIR: derivada de órdenes
UnidadesPedidas,False,88,53,float64,EXCLUIR: derivada de órdenes


### Revisión de los proveedores 1699 y 1700 y de las variables candidatas
**Qué hace la celda siguiente:** muestra los datos de los proveedores 1699 y 1700 y cruza cada variable candidata con el segmento (incluye la columna `SIN_ORDENES` para los 18).

**Qué significa:** los dos proveedores quedaron en *Ocasionales*, con 3 y 1 órdenes por 83.10 y 17.00 en total, nombres de ejemplo y fechas de modificación de 2026: son registros de prueba. Ambos tienen `CreditRating = 3`, lo que hace que ese valor parezca asociado a *Ocasionales* por un artefacto. Los cruces muestran además que ninguna de las tres variables separa claramente los segmentos.

In [18]:
print(df[df.VendorID.isin([1699,1700])][['VendorID','NombreProveedor','CreditRating','NumeroOrdenes','ValorTotalOrdenes','Segmento']])
for c in ['CreditRating','PreferredVendorStatus','ActiveFlag']:
    print(pd.crosstab(df[c], df['Segmento'].fillna('SIN_ORDENES')), '\n')

     VendorID             NombreProveedor  CreditRating  NumeroOrdenes  \
104      1699  Distribuidora Ejemplo S.A.             3            3.0   
105      1700                    garfield             3            1.0   

     ValorTotalOrdenes     Segmento  
104               83.1  Ocasionales  
105               17.0  Ocasionales  
Segmento      Grandes recurrentes  Ocasionales  Pequeños recurrentes  \
CreditRating                                                           
1                              31            7                    30   
2                               5            0                     4   
3                               2            2                     5   
4                               1            0                     0   
5                               1            0                     0   

Segmento      SIN_ORDENES  
CreditRating               
1                      16  
2                       0  
3                       0  
4                 

### Exclusión de los registros de prueba
**Qué hace la celda siguiente:** quita a los proveedores 1699 y 1700 del conjunto de modelado y vuelve a mostrar los conteos.

**Qué significa:** al ser datos de prueba, no representan proveedores reales y su inclusión sesgaría el modelo (por ejemplo, sus direcciones fuera de Estados Unidos coinciden con el segmento *Ocasionales* solo por casualidad). Quedan 104 proveedores: 86 para entrenar (40 Grandes, 39 Pequeños, 7 Ocasionales) y 18 para predecir. La segmentación original no se modifica; la exclusión solo aplica a este notebook.

In [19]:
PRUEBA = [1699, 1700]   # registros de prueba en la base: se excluyen del modelado
df = df[~df['VendorID'].isin(PRUEBA)].reset_index(drop=True)
print(df['conjunto'].value_counts()); print(df['Segmento'].value_counts(dropna=False))

conjunto
entrenamiento    86
predecir         18
Name: count, dtype: int64
Segmento
Grandes recurrentes     40
Pequeños recurrentes    39
NaN                     18
Ocasionales              7
Name: count, dtype: int64


## TASK 6.1.2 – Atributos adicionales del proveedor
Con tres predictoras (una casi constante) el modelo difícilmente superará la línea base, por lo que se consultaron atributos adicionales en SQL Server con `src/extract_vendor_attributes.py`: ubicación (`Person.BusinessEntityAddress` → `Address` → `StateProvince` → `CountryRegion`), `AccountNumber` y `PurchasingWebServiceURL`.

**Qué hace la celda siguiente:** carga `vendor_attributes.csv`, comprueba que cubre a todos los proveedores del modelado, y muestra la distribución por país y provincia y su cruce con el segmento.

**Qué significa:** el país no varía (todos los proveedores de modelado son de Estados Unidos); hay 23 provincias pero solo California, Washington y Oregón tienen volumen, y su reparto entre segmentos es parecido al general; solo unos pocos proveedores tienen URL. `AccountNumber` es un identificador y se descarta.

In [20]:
attr = pd.read_csv(ROOT/'data/raw/vendor_attributes.csv')
attr = attr[attr['VendorID'].isin(df['VendorID'])]
print('Proveedores con atributos:', attr['VendorID'].nunique(), 'de', len(df))
print(attr['Pais'].value_counts()); print('Provincias distintas:', attr['Provincia'].nunique())
tmp = attr.merge(df[['VendorID','Segmento']], on='VendorID')
tmp['Segmento'] = tmp['Segmento'].fillna('SIN_ORDENES')
print(pd.crosstab(tmp['Provincia'], tmp['Segmento']))
print(pd.crosstab(tmp['PurchasingWebServiceURL'].notna().rename('TieneURL'), tmp['Segmento']))

Proveedores con atributos: 104 de 104
Pais
United States    104
Name: count, dtype: int64
Provincias distintas: 21
Segmento       Grandes recurrentes  Ocasionales  Pequeños recurrentes  \
Provincia                                                               
Arizona                          1            0                     0   
California                      20            3                    12   
Colorado                         0            1                     0   
Georgia                          1            0                     0   
Idaho                            0            0                     2   
Illinois                         1            0                     0   
Massachusetts                    1            0                     0   
Michigan                         0            0                     1   
Missouri                         1            0                     0   
Montana                          1            0                     2   
Nebraska 

**Qué hace la celda siguiente:** crea dos variables nuevas a partir de esos atributos: `ProvinciaGrupo` (California, Washington, Oregón y «Otras», porque el resto de provincias tiene muy pocos proveedores) y `TieneURL` (1 si el proveedor tiene URL de compras). Las une al conjunto de modelado y define la lista final de predictoras, `FEATURES`.

**Qué significa:** son las variables que probarán los modelos. El conteo de nulos debe dar 0 y 0: todos los proveedores tienen dirección. Si se quisiera probar otra variable, se agrega aquí y el resto del notebook se adapta al volver a ejecutarlo.

In [21]:
attr['ProvinciaGrupo'] = np.where(attr['Provincia'].isin(['California','Washington','Oregon']), attr['Provincia'], 'Otras')
attr['TieneURL'] = attr['PurchasingWebServiceURL'].notna().astype(int)
df = df.merge(attr[['VendorID','ProvinciaGrupo','TieneURL']], on='VendorID', how='left')
print(df[['ProvinciaGrupo','TieneURL']].isna().sum())   # debe dar 0 y 0
FEATURES_BASE = ['CreditRating','PreferredVendorStatus','ActiveFlag']
FEATURES_EXTRA = ['ProvinciaGrupo','TieneURL']
FEATURES = FEATURES_BASE + FEATURES_EXTRA

ProvinciaGrupo    0
TieneURL          0
dtype: int64


## TASK 6.1.3 – Dataset de modelado
**Qué hace la celda siguiente:** construye la matriz de predictoras, convierte los booleanos a 0/1 y la separa en dos grupos: `X_train` y `y_train` (proveedores con segmento, con sus `VendorID` en `ids_train`) y `X_pred` (los 18 a predecir, con sus identificadores en `ids_pred`).

**Qué significa:** el primer grupo se usa para entrenar y evaluar, y los 18 solo se usan al final para obtener la predicción. La codificación de las variables categóricas (`CreditRating`, `ProvinciaGrupo`) no se hace aquí, sino dentro del `Pipeline` de cada modelo, para que se ajuste solo con los datos de entrenamiento de cada partición y no se filtre información de la partición de prueba (*sin fuga de datos*).

In [22]:
X_all = df[FEATURES].copy()
X_all['PreferredVendorStatus'] = X_all['PreferredVendorStatus'].astype(int)
X_all['ActiveFlag'] = X_all['ActiveFlag'].astype(int)
ent = df['conjunto']=='entrenamiento'
X_train, y_train = X_all[ent].reset_index(drop=True), df.loc[ent,'Segmento'].reset_index(drop=True)
ids_train = df.loc[ent,'VendorID'].reset_index(drop=True)
X_pred = X_all[~ent].reset_index(drop=True); ids_pred = df.loc[~ent,['VendorID','NombreProveedor']].reset_index(drop=True)
print(X_train.shape, X_pred.shape); print(y_train.value_counts())

(86, 5) (18, 5)
Segmento
Grandes recurrentes     40
Pequeños recurrentes    39
Ocasionales              7
Name: count, dtype: int64


## TASK 6.1.4 – Validación y línea base
**Qué hace la celda siguiente:** calcula el tamaño de la clase más pequeña, define la validación cruzada estratificada repetida (3 particiones repetidas 20 veces), crea la función `evaluar` (que entrena y evalúa un modelo en cada partición y guarda las predicciones de cada una) y evalúa dos modelos de referencia: uno que siempre predice la clase más frecuente y otro que predice al azar respetando la proporción de clases. Reporta precisión y F1 macro con su desviación estándar.

**Qué significa:** *estratificada* mantiene la proporción de segmentos en cada partición; el número de particiones no puede superar el tamaño de la clase más pequeña (por eso 3). La **línea base** es el mínimo que un modelo debe superar para demostrar que aprende algo útil. El *F1 macro* promedia el F1 de cada segmento por igual, por lo que no se deja engañar por la clase mayoritaria.

In [23]:
n_min = y_train.value_counts().min(); print('Clase más pequeña:', n_min)
cv = RepeatedStratifiedKFold(n_splits=3, n_repeats=20, random_state=SEED)   # n_splits <= n_min
OOF = []   # predicciones de cada partición (para la HU 6.3)
def evaluar(modelo, X, y, cv, nombre=None):
    accs, f1s = [], []
    nsp = cv.cvargs['n_splits']
    for i, (tr, te) in enumerate(cv.split(X, y)):
        m = modelo.fit(X.iloc[tr], y.iloc[tr]); p = m.predict(X.iloc[te])
        accs.append(accuracy_score(y.iloc[te], p)); f1s.append(f1_score(y.iloc[te], p, average='macro'))
        if nombre:
            OOF.append(pd.DataFrame({'modelo':nombre,'repeticion':i//nsp,'pliegue':i%nsp,'VendorID':ids_train.iloc[te].values,'y_real':y.iloc[te].values,'y_pred':p}))
    return pd.Series({'accuracy_media':np.mean(accs),'accuracy_std':np.std(accs),'f1_macro_media':np.mean(f1s),'f1_macro_std':np.std(f1s)})
base = pd.DataFrame({
 'mayoritaria': evaluar(DummyClassifier(strategy='most_frequent'), X_train, y_train, cv),
 'estratificado': evaluar(DummyClassifier(strategy='stratified', random_state=SEED), X_train, y_train, cv)}).T
base

Clase más pequeña: 7


,accuracy_media,accuracy_std,f1_macro_media,f1_macro_std
mayoritaria,0.465107,0.014089,0.211595,0.004373
estratificado,0.420977,0.077523,0.323377,0.078234


## HU 6.2 – Entrenamiento de los modelos (TASK 6.2.1 y 6.2.2)
**Qué hace la celda siguiente:** define y evalúa los dos modelos: regresión logística (TASK 6.2.1) y Random Forest (TASK 6.2.2). Cada uno es un `Pipeline` (preparación de variables + modelo) con una búsqueda de hiperparámetros (`GridSearchCV`) dentro de una validación interna, y se evalúa con una validación externa (validación cruzada anidada). Para que la comparación sea justa, las líneas base se vuelven a evaluar con **exactamente las mismas particiones** que los modelos (`cv_ext`, que son las 3 primeras repeticiones de `cv`). Muestra la tabla con las líneas base y los dos modelos.

**Qué significa:** la validación anidada evita que el ajuste de hiperparámetros infle el resultado. `class_weight='balanced'` da más peso al segmento menos frecuente. Para saber si un modelo sirve, se compara su precisión y su F1 macro contra las filas de línea base de la tabla.

In [24]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold

def prep(escalar):
    num = [c for c in ['PreferredVendorStatus','ActiveFlag','TieneURL'] if c in FEATURES]
    cat = [c for c in ['CreditRating','ProvinciaGrupo'] if c in FEATURES]
    return ColumnTransformer([('cat', OneHotEncoder(handle_unknown='ignore'), cat),
                              ('num', StandardScaler() if escalar else 'passthrough', num)])
inner = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
lr = GridSearchCV(Pipeline([('prep',prep(True)),('clf',LogisticRegression(max_iter=2000, class_weight='balanced'))]),
                  {'clf__C':[0.01,0.1,1,10]}, cv=inner, scoring='f1_macro')
rf = GridSearchCV(Pipeline([('prep',prep(False)),('clf',RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=SEED, n_jobs=-1))]),
                  {'clf__max_depth':[2,None],'clf__min_samples_leaf':[1,5]}, cv=inner, scoring='f1_macro')
cv_ext = RepeatedStratifiedKFold(n_splits=3, n_repeats=3, random_state=SEED)  # = primeras 3 repeticiones de cv
OOF.clear()
base_ext = pd.DataFrame({
 'mayoritaria': evaluar(DummyClassifier(strategy='most_frequent'), X_train, y_train, cv_ext, 'mayoritaria'),
 'estratificado': evaluar(DummyClassifier(strategy='stratified', random_state=SEED), X_train, y_train, cv_ext, 'estratificado')}).T
res = pd.DataFrame({'regresion_logistica': evaluar(lr, X_train, y_train, cv_ext, 'regresion_logistica'),
                    'random_forest': evaluar(rf, X_train, y_train, cv_ext, 'random_forest')}).T
tabla = pd.concat([base_ext, res])
tabla

,accuracy_media,accuracy_std,f1_macro_media,f1_macro_std
mayoritaria,0.465107,0.014089,0.211595,0.004373
estratificado,0.422140,0.080472,0.319746,0.059255
regresion_logistica,0.293788,0.074699,0.263348,0.068157
random_forest,0.293788,0.068772,0.247902,0.059318


### Comparación automática contra la línea base
**Qué hace la celda siguiente:** compara cada modelo con las líneas base y escribe una frase por modelo.

**Qué significa:** un modelo solo aporta información si supera a la clase mayoritaria en precisión **y** al azar estratificado en F1 macro. La frase impresa indica si lo logra o no con los datos de esta ejecución.

In [25]:
for m in ['regresion_logistica','random_forest']:
    a = tabla.loc[m,'accuracy_media'] - tabla.loc['mayoritaria','accuracy_media']
    f = tabla.loc[m,'f1_macro_media'] - tabla.loc['estratificado','f1_macro_media']
    ok = (a > 0) and (f > 0)
    print(f"{m}: precisión {a:+.3f} vs mayoritaria; F1 macro {f:+.3f} vs estratificado -> "
          + ("SUPERA ambas líneas base." if ok else "NO supera ambas líneas base."))

regresion_logistica: precisión -0.171 vs mayoritaria; F1 macro -0.056 vs estratificado -> NO supera ambas líneas base.
random_forest: precisión -0.171 vs mayoritaria; F1 macro -0.072 vs estratificado -> NO supera ambas líneas base.


## Conclusiones (HU 6.1 y HU 6.2)
1. **Variables.** Solo `CreditRating`, `PreferredVendorStatus` y `ActiveFlag` existen para los 106 proveedores sin derivarse de las órdenes. Las nueve variables numéricas del consolidado se excluyen: seis se calculan con órdenes (fuga de datos) y todas están vacías para los 18 proveedores sin órdenes, que además no aparecen en `ProductVendor`.
2. **Registros de prueba.** Los proveedores 1699 y 1700 son datos de prueba (nombres de ejemplo, direcciones en Alemania y Australia, fechas de 2026, 3 y 1 órdenes por montos ínfimos). Cayeron en *Ocasionales* y hacían que `CreditRating = 3` y «fuera de EE. UU.» parecieran señales. Se excluyeron del modelado; quedan 86 proveedores de entrenamiento (40 Grandes recurrentes, 39 Pequeños recurrentes y 7 Ocasionales) y 18 por predecir.
3. **Atributos adicionales (TASK 6.1.2).** El país no varía (todos de Estados Unidos). La provincia tiene 23 valores, pero solo California, Washington y Oregón tienen volumen, y su reparto entre segmentos se parece al general. Solo unos pocos proveedores tienen URL. `AccountNumber` es un identificador. Se probaron `ProvinciaGrupo` y `TieneURL`, sin evidencia previa de que separen los segmentos.
4. **Validación (TASK 6.1.4).** Validación cruzada estratificada de 3 particiones (el máximo permitido por los 7 Ocasionales), repetida 20 veces para las líneas base y 3 veces para la comparación con los modelos, con las mismas particiones y semilla. La clase mayoritaria representa 40 de 86 proveedores (46,5 %), y esa es la precisión que un modelo debe superar.
5. **Modelos (TASK 6.2.1 y 6.2.2).** Regresión logística y Random Forest, cada uno en un `Pipeline` con codificación dentro de cada partición y ajuste de hiperparámetros anidado, con `class_weight='balanced'`.
6. **Lectura de resultados.** Se deben leer en la tabla comparativa y en la frase automática de arriba. Si ningún modelo supera ambas líneas base, la conclusión es que los atributos disponibles del proveedor (calificación, estado preferido, estado activo, provincia, URL) **no permiten predecir su segmento**, que se define por monto y número de órdenes. En ese caso, cualquier segmento asignado a los 18 proveedores sin órdenes debe tomarse como de baja confianza.
7. **Limitaciones.** Solo hay 7 Ocasionales reales, así que las métricas de esa clase son muy ruidosas. Los 18 proveedores sin órdenes no tienen actividad de compra, por lo que asignarles un segmento es una extrapolación, y casi todos tienen `CreditRating = 1` y `PreferredVendorStatus = True`. El segmento proviene de una agrupación K-Means de dos variables, no de una clasificación formal.

## Entrega para la HU 6.3
**Qué hace la celda siguiente:** guarda en `data/processed/` los insumos de la HU 6.3: el conjunto de entrenamiento (`modelado_train.csv`), el conjunto a predecir (`modelado_pred.csv`), las predicciones de cada partición de validación de todos los modelos y líneas base (`oof_predicciones.csv`), la tabla comparativa (`resultados_validacion.csv`) y los dos modelos finales entrenados con todos los datos de entrenamiento (`modelos_epic6.joblib`, con la lista de variables, la semilla y la configuración de validación). Imprime los hiperparámetros elegidos.

**Qué significa:** con estos archivos, la HU 6.3 puede calcular la matriz de confusión, comparar los modelos y aplicar el mejor a los 18 proveedores sin repetir el entrenamiento. Estos archivos no se suben a Git (la carpeta `data/` está excluida): se comparten directamente.

In [26]:
import joblib
OUT = PROC
X_train.assign(VendorID=ids_train, Segmento=y_train).to_csv(OUT/'modelado_train.csv', index=False)
X_pred.assign(VendorID=ids_pred.VendorID, NombreProveedor=ids_pred.NombreProveedor).to_csv(OUT/'modelado_pred.csv', index=False)
pd.concat(OOF, ignore_index=True).to_csv(OUT/'oof_predicciones.csv', index=False)
tabla.to_csv(OUT/'resultados_validacion.csv')
lr.fit(X_train, y_train); rf.fit(X_train, y_train)
joblib.dump({'lr':lr.best_estimator_, 'rf':rf.best_estimator_, 'features':FEATURES, 'seed':SEED,
             'cv':{'n_splits':3,'n_repeats':3,'random_state':SEED}, 'excluidos':PRUEBA}, OUT/'modelos_epic6.joblib')
print(lr.best_params_, rf.best_params_)

{'clf__C': 10} {'clf__max_depth': 2, 'clf__min_samples_leaf': 1}
